Imports the analysis libraries and local test helpers, and sets the requested model dtype to bfloat16 for the experiments. The `device` variable selects MPS or CPU, while model placement uses `device_map="auto"`. Saved outputs are historical, not results of this editorial update.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from jaxtyping import Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import course_tests as tests

device = t.device("mps" if t.mps.is_available() else "cpu")
dtype = t.bfloat16

MAIN = __name__ == "__main__"


Resolves the dataset directory relative to the current working directory and checks that it exists. Subsequent CSV reads use this path.


In [ ]:
# Run the notebook with the project directory as its working directory.
PROJECT_ROOT = Path.cwd()
GOT_DATASETS = PROJECT_ROOT / "datasets"
assert GOT_DATASETS.is_dir(), f"Run from the project root: missing {GOT_DATASETS}"


Checks that `HF_TOKEN` is exported for access to the gated model. The value is not passed explicitly to the loader; authentication relies on the model library’s environment handling.


In [ ]:
# Export HF_TOKEN before starting Jupyter; credentials are not bundled.
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please export HF_TOKEN for access to meta-llama/Llama-2-13b-hf"


Loads `meta-llama/Llama-2-13b-hf` with 8-bit quantization, requested bfloat16 dtype, automatic device placement, and EOS-based right padding. The printed dimensions and layer settings define the later extraction and intervention experiments; no model revision is pinned.


In [ ]:
MODEL_NAME = "meta-llama/Llama-2-13b-hf"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    dtype=dtype,
    device_map="auto",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size
# Layer choices from the geometry-of-truth repo config for llama-2-13b. The paper
# found truth representations are concentrated in early-to-mid layers, and identified
# these specific layers via patching experiments (Section 3, "group (b)").
PROBE_LAYER = 14
INTERVENE_LAYER = 8

print(f"Model: {MODEL_NAME}")
print(f"Layers: {NUM_LAYERS}, Hidden dim: {D_MODEL}")
print(f"Probe layer: {PROBE_LAYER}, Intervene layer: {INTERVENE_LAYER}")

Loads `cities`, `sp_en_trans`, and `larger_than` CSVs for the probe experiments, with labels 1 for true and 0 for false. Counts and sample rows show class balance and statement format.


In [ ]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(GOT_DATASETS / f"{name}.csv")
    datasets[name] = df
    print(f"\n{name}: {len(df)} statements ({df['label'].sum()} true, {(1 - df['label']).sum():.0f} false)")
    display(df.head(4))

Defines batched extraction of the last non-padding token’s hidden state, using `layer + 1` to account for the embedding entry in `hidden_states`. Returned tensors are moved to CPU for later analysis, and the local test checks the extractor against the loaded model.


In [ ]:
def extract_activations(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
    batch_size: int = 25,
) -> dict[int, Float[Tensor, "n_statements d_model"]]:
    """
    Extract last-token hidden state activations from specified layers for a list of statements.

    Args:
        statements: List of text statements to process.
        model: A HuggingFace causal language model.
        tokenizer: The corresponding tokenizer.
        layers: List of layer indices (0-indexed) to extract activations from.
        batch_size: Number of statements to process at once.

    Returns:
        Dictionary mapping layer index to tensor of activations, shape [n_statements, d_model].
    """
    activations = {layer: [] for layer in layers}
    for i in range(0, len(statements), batch_size):
        inputs = tokenizer(
            statements[i:i + batch_size],
            padding=True,
            return_tensors="pt"
        ).to(model.device)
        with t.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        attention_mask = inputs["attention_mask"]
        last_token_indices = attention_mask.sum(dim=1) - 1
        batch_indices = t.arange(0, inputs["input_ids"].shape[0])
        for layer in layers:
            hidden_state = outputs.hidden_states[layer + 1]
            last_token_activations = hidden_state[batch_indices, last_token_indices]
            activations[layer].append(last_token_activations.cpu())

    for layer in layers:
        activations[layer] = t.cat(activations[layer], dim=0)

    return activations
        
    raise NotImplementedError()


tests.test_extract_activations(extract_activations, model, tokenizer, PROBE_LAYER, D_MODEL)

Extracts layer-14 activations for all three datasets and stores aligned statements and labels for plotting and probing. The summary reports class counts, activation shapes, and mean vector norms.


In [ ]:
# Extract activations at the probe layer for all datasets
activations = {}
labels_dict = {}
statements_dict = {}

for name in DATASET_NAMES:
    df = datasets[name]
    statements = df["statement"].tolist()
    labs = t.tensor(df["label"].values, dtype=t.float32)
    statements_dict[name] = statements

    acts = extract_activations(statements, model, tokenizer, [PROBE_LAYER])
    activations[name] = acts[PROBE_LAYER]
    labels_dict[name] = labs

# Show summary table
summary = pd.DataFrame(
    {
        "Dataset": DATASET_NAMES,
        "N statements": [len(datasets[n]) for n in DATASET_NAMES],
        "N true": [int(datasets[n]["label"].sum()) for n in DATASET_NAMES],
        "N false": [int((1 - datasets[n]["label"]).sum()) for n in DATASET_NAMES],
        "Act shape": [str(tuple(activations[n].shape)) for n in DATASET_NAMES],
        "Mean norm": [f"{activations[n].norm(dim=-1).mean():.1f}" for n in DATASET_NAMES],
    }
)
display(summary)

Computes principal components by mean-centering float32 activations and eigendecomposing their full covariance matrix, then calls the local PCA test. The returned columns are the highest-variance directions; forming the full matrix can require substantial memory.


In [ ]:
def get_pca_components(
    activations: Float[Tensor, "n d_model"],
    k: int = 2,
) -> Float[Tensor, "d_model k"]:
    """
    Compute the top-k principal components of the activation matrix.

    Args:
        activations: Activation matrix, shape [n_samples, d_model].
        k: Number of principal components to return.

    Returns:
        Matrix of top-k eigenvectors as columns, shape [d_model, k].
    """
    activations = activations.float()
    X = activations - activations.mean(dim=0)

    cov = X.t() @ X / (X.shape[0] - 1)

    eigenvalues, eigenvectors = t.linalg.eigh(cov)

    sorted_indices = t.argsort(eigenvalues, descending=True)
    top_k = eigenvectors[:, sorted_indices[:k]]

    return top_k
    raise NotImplementedError()


tests.test_get_pca_components(get_pca_components, activations["cities"].float(), D_MODEL)

Projects each dataset independently onto its first two principal components to inspect label separation without using labels to fit the projection. Blue and red points denote true and false statements, and axis percentages report explained variance; these two-dimensional views do not establish full-space classification accuracy.


In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=DATASET_NAMES)

for i, name in enumerate(DATASET_NAMES):
    acts = activations[name].float()
    labs = labels_dict[name]

    label_text = ["True" if l == 1 else "False" for l in labs.tolist()]

    prompts = statements_dict[name]
    pcs = get_pca_components(acts, k=2)
    X_centered = acts - acts.mean(dim=0)
    projected = (X_centered @ pcs).numpy()

    # Compute variance explained
    total_var = X_centered.var(dim=0).sum().item()
    pc_var = t.tensor(projected).var(dim=0)
    pct_explained = (pc_var / total_var * 100).tolist()

    colors = ["blue" if l == 1 else "red" for l in labs.tolist()]
    fig.add_trace(
        go.Scatter(
            x=projected[:, 0],
            y=projected[:, 1],
            mode="markers",
            marker=dict(color=colors, size=3, opacity=0.5),
            name=name,
            showlegend=False,
            hovertext=prompts,
            customdata=list(zip(prompts, label_text)),
            hovertemplate=(
                "<b>%{customdata[1]}</b><br>"
                "%{customdata[0]}<br>"
                "PC1: %{x:.2f}<br>"
                "PC2: %{y:.2f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=i + 1,
    )
    fig.update_xaxes(title_text=f"PC1 ({pct_explained[0]:.1f}%)", row=1, col=i + 1)
    fig.update_yaxes(title_text=f"PC2 ({pct_explained[1]:.1f}%)", row=1, col=i + 1)

# Add a legend manually
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="blue", size=8), name="True"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="red", size=8), name="False"))

fig.update_layout(
    title="PCA of Truth Representations (Layer 14, Last Token)",
    height=400,
    width=1200,
)
fig.show()

Sweeps all model layers on `cities` using a seed-42 random 80/20 split and a difference-of-means classifier centered at the class midpoint. The table and curves compare train/test accuracy by layer; the reported best layer is selected on this test split, without an independent validation set.


In [ ]:
def layer_sweep_accuracy(
    statements: list[str],
    labels: Float[Tensor, " n"],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
    train_frac: float = 0.8,
    batch_size: int = 25,
) -> dict[str, list[float]]:
    """
    For each layer, train a difference-of-means classifier and compute train/test accuracy.

    Args:
        statements: List of statements.
        labels: Binary labels (1=true, 0=false).
        model: The language model.
        tokenizer: The tokenizer.
        layers: List of layer indices to sweep over.
        train_frac: Fraction of data for training.
        batch_size: Batch size for activation extraction.

    Returns:
        Dict with keys "train_acc" and "test_acc", each a list of accuracies per layer.
    """
    n_train = int(train_frac * len(statements))
    perm = t.randperm(len(statements))
    train_idx = perm[:n_train]
    test_idx = perm[n_train:]

    train_statements = [statements[i] for i in train_idx]
    test_statements = [statements[i] for i in test_idx]
    train_labels = labels[train_idx]
    test_labels = labels[test_idx]
    
    train_acts = extract_activations(train_statements, model, tokenizer, layers, batch_size)
    test_acts = extract_activations(test_statements, model, tokenizer, layers, batch_size)

    train_accuracies = []
    test_accuracies = []

    for layer in layers:
        tr_acts = train_acts[layer]
        te_acts = test_acts[layer]

        mean_act_true = tr_acts[train_labels == 1].mean(dim=0)
        mean_act_false = tr_acts[train_labels == 0].mean(dim=0)
        midpoint = (mean_act_false + mean_act_true) / 2
        direction = mean_act_true - mean_act_false

        train_preds = (((tr_acts - midpoint) @ direction) > 0).float()
        test_preds = (((te_acts - midpoint) @ direction) > 0).float()

        train_accuracy = (train_preds == train_labels).float().mean().item()
        test_accuracy = (test_preds == test_labels).float().mean().item()
        train_accuracies.append(train_accuracy)
        test_accuracies.append(test_accuracy)

    results = {"train_acc": train_accuracies, "test_acc": test_accuracies}
    return results

    raise NotImplementedError()


t.manual_seed(42)
all_layers = list(range(NUM_LAYERS))
cities_statements = datasets["cities"]["statement"].tolist()
cities_labels = t.tensor(datasets["cities"]["label"].values, dtype=t.float32)

sweep_results = layer_sweep_accuracy(cities_statements, cities_labels, model, tokenizer, all_layers)

# Print results as a table
sweep_df = pd.DataFrame(
    {
        "Layer": all_layers,
        "Train Acc": [f"{a:.3f}" for a in sweep_results["train_acc"]],
        "Test Acc": [f"{a:.3f}" for a in sweep_results["test_acc"]],
    }
)
display(sweep_df)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["train_acc"], mode="lines+markers", name="Train"))
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["test_acc"], mode="lines+markers", name="Test"))
fig.add_vline(x=PROBE_LAYER, line_dash="dash", line_color="gray", annotation_text=f"Probe layer ({PROBE_LAYER})")
fig.update_layout(
    title="Layer Sweep: Difference-of-Means Accuracy on Cities Dataset",
    xaxis_title="Layer",
    yaxis_title="Accuracy",
    yaxis_range=[0.4, 1.05],

    height=400,
    width=800,
)
fig.show()

best_layer = all_layers[int(np.argmax(sweep_results["test_acc"]))]
print(f"\nBest layer by test accuracy: {best_layer} ({max(sweep_results['test_acc']):.3f})")
print(f"Configured probe layer: {PROBE_LAYER} ({sweep_results['test_acc'][PROBE_LAYER]:.3f})")

Creates seed-42 random 80/20 splits of each dataset’s layer-14 activations and labels for subsequent probe comparisons. Printed sizes confirm the partition counts; the split is neither stratified nor grouped by statement content.


In [ ]:
# Create train/test splits for all datasets
t.manual_seed(42)
train_acts, test_acts = {}, {}
train_labels, test_labels = {}, {}

for name in DATASET_NAMES:
    acts = activations[name]
    labs = labels_dict[name]
    n = len(acts)
    perm = t.randperm(n)
    n_train = int(0.8 * n)

    train_acts[name] = acts[perm[:n_train]]
    test_acts[name] = acts[perm[n_train:]]
    train_labels[name] = labs[perm[:n_train]]
    test_labels[name] = labs[perm[n_train:]]

    print(f"{name}: train={n_train}, test={n - n_train}")

Defines a mass-mean (MM) probe from class-mean differences and pooled within-class covariance, then reports cities train/test accuracy. Default predictions use an uncentered dot product, unlike the layer sweep; `iid=True` uses the covariance pseudoinverse, which the constructor requires even though its argument defaults to `None`.


In [ ]:
class MMProbe(t.nn.Module):
    def __init__(
        self,
        direction: Float[Tensor, " d_model"],
        covariance: Float[Tensor, "d_model d_model"] | None = None,
        atol: float = 1e-3,
    ):
        super().__init__()
        # Store direction and precompute inverse covariance
        self.direction = direction
        self.covariance = covariance
        self.inv_cov = t.nn.Parameter(t.linalg.pinv(covariance.float(), hermitian=True, atol=atol), requires_grad=False)

    def forward(self, x: Float[Tensor, "n d_model"], iid: bool = False) -> Float[Tensor, " n"]:
        if iid == False and self.inv_cov is not None:
            return t.sigmoid(x @ self.direction)
        return t.sigmoid(x @ self.inv_cov @ self.direction)
        raise NotImplementedError()

    def pred(self, x: Float[Tensor, "n d_model"], iid: bool = False) -> Float[Tensor, " n"]:
        return self(x, iid=iid).round()

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        device: str = "cpu",
    ) -> "MMProbe":
        activations_true = acts[labels == 1]
        activations_false = acts[labels == 0]
        mean_activations_true = activations_true.mean(dim=0)
        mean_activations_false = activations_false.mean(dim=0)

        direction = mean_activations_true - mean_activations_false
        X = t.cat((activations_true - mean_activations_true, activations_false - mean_activations_false), dim=0) 
        covariance = (X.T @ X) / (acts.shape[0])
        return MMProbe(direction, covariance).to(device)
        raise NotImplementedError()


mm_probe = MMProbe.from_data(train_acts["cities"], train_labels["cities"])

# Train accuracy
train_preds = mm_probe.pred(train_acts["cities"])
train_acc = (train_preds == train_labels["cities"]).float().mean().item()

# Test accuracy
test_preds = mm_probe.pred(test_acts["cities"])
test_acc = (test_preds == test_labels["cities"]).float().mean().item()
assert test_acc > 0.69, "Expected at least 69% accuracy"

print("MMProbe on cities:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  Direction norm: {mm_probe.direction.norm().item():.3f}")
print(f"  Direction (first 5): {mm_probe.direction[:5].tolist()}")

Fits logistic regression (LR) without an intercept on `StandardScaler`-normalized activations, using `C=0.1`, then compares held-out MM/LR accuracies and cities direction similarity. LR directions are expressed in standardized coordinates, so their cosine similarity with raw-coordinate MM directions has that limitation; the `device` argument is unused.


In [ ]:
class LRProbe(t.nn.Module):
    def __init__(self, d_in: int, scaler_mean: Tensor | None = None, scaler_scale: Tensor | None = None):
        super().__init__()
        self.net = t.nn.Sequential(
            t.nn.Linear(d_in, 1, bias=False),
            t.nn.Sigmoid()
        )
        self.scaler_mean = scaler_mean
        self.scaler_scale = scaler_scale

    def _normalize(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, "n d_model"]:
        """Apply StandardScaler normalization if scaler parameters are available."""
        if self.scaler_mean is not None and self.scaler_scale is not None:
            return (x - self.scaler_mean) / self.scaler_scale
        return x

    def forward(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        normalized_x = self._normalize(x)
        probe = self.net(normalized_x)
        return probe.squeeze(-1)
        raise NotImplementedError()

    def pred(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self(x).round()

    @property
    def direction(self) -> Float[Tensor, " d_model"]:
        return self.net[0].weight.data[0]

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        C: float = 0.1,
        device: str = "cpu",
    ) -> "LRProbe":
        """
        Train an LR probe using sklearn's LogisticRegression with StandardScaler normalization.

        Args:
            acts: Activation matrix [n_samples, d_model].
            labels: Binary labels (1=true, 0=false).
            C: Inverse regularization strength (lower = stronger regularization).
                Default 0.1 (reg_coeff=10) matches the deception-detection paper's cfg.yaml.
                The repo class default is reg_coeff=1000 (C=0.001), which is stronger.
            device: Device to place the resulting probe on.
        """
        scaler = StandardScaler()
        model = LogisticRegression(C=C, fit_intercept=False)
        acts_normalized = scaler.fit_transform(acts.float().cpu().numpy())
        model.fit(acts_normalized, labels.float().cpu().numpy())

        scaler_mean = t.tensor(scaler.mean_, dtype=t.float32)
        scaler_scale = t.tensor(scaler.scale_, dtype=t.float32)

        probe = LRProbe(acts.shape[1], scaler_mean, scaler_scale)
        probe.net[0].weight.data[0] = t.tensor(model.coef_)
        return probe
        raise NotImplementedError()


lr_probe = LRProbe.from_data(train_acts["cities"], train_labels["cities"], device="cpu")

# Train accuracy
train_preds = lr_probe.pred(train_acts["cities"])
train_acc = (train_preds == train_labels["cities"]).float().mean().item()

# Test accuracy
test_preds = lr_probe.pred(test_acts["cities"])
test_acc = (test_preds == test_labels["cities"]).float().mean().item()

print("LRProbe on cities:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  Direction norm: {lr_probe.direction.norm().item():.3f}")
assert test_acc >= 0.90, f"Test accuracy too low: {test_acc:.3f} (expected >= 0.90)"

# Compare directions
mm_dir = (mm_probe.direction / mm_probe.direction.norm()).float()
lr_dir = lr_probe.direction / lr_probe.direction.norm()
cos_sim = (mm_dir @ lr_dir).item()
print(f"\nCosine similarity between MM and LR directions: {cos_sim:.4f}")

# Compare both probes across all 3 datasets
results_rows = []
for name in DATASET_NAMES:
    mm_p = MMProbe.from_data(train_acts[name], train_labels[name])
    lr_p = LRProbe.from_data(train_acts[name], train_labels[name])

    mm_test_acc = (mm_p.pred(test_acts[name]) == test_labels[name]).float().mean().item()
    lr_test_acc = (lr_p.pred(test_acts[name]) == test_labels[name]).float().mean().item()
    results_rows.append({"Dataset": name, "MM Test Acc": f"{mm_test_acc:.3f}", "LR Test Acc": f"{lr_test_acc:.3f}"})

results_df = pd.DataFrame(results_rows)
print("\nProbe accuracy comparison across datasets:")
display(results_df)

# Bar chart
fig = go.Figure()
fig.add_trace(go.Bar(name="MMProbe", x=DATASET_NAMES, y=[float(r["MM Test Acc"]) for r in results_rows]))
fig.add_trace(go.Bar(name="LRProbe", x=DATASET_NAMES, y=[float(r["LR Test Acc"]) for r in results_rows]))
fig.update_layout(
    title="Probe Test Accuracy by Dataset",
    yaxis_title="Test Accuracy",
    yaxis_range=[0.5, 1.05],
    barmode="group",
    height=400,
    width=600,
)
fig.show()

Fits each probe type on each training dataset and evaluates it on every test dataset to measure transfer. Heatmap rows identify training data and columns identify test data, so off-diagonal entries measure cross-dataset accuracy; printed LR direction similarities compare coefficients from separately fitted scalers.


In [ ]:
def compute_generalization_matrix(
    train_acts: dict[str, Float[Tensor, "n d"]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, Float[Tensor, "n d"]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
) -> Float[Tensor, "n_datasets n_datasets"]:
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    gen_mat = t.zeros(len(dataset_names), len(dataset_names))
    for i, train_dataset in enumerate(dataset_names):
        probe = probe_cls.from_data(train_acts[train_dataset], train_labels[train_dataset])
        
        for j, test_dataset in enumerate(dataset_names):
            preds = probe.pred(test_acts[test_dataset])
            acc = (preds == test_labels[test_dataset]).float().mean().item()
            gen_mat[i, j] = acc

    return gen_mat
    raise NotImplementedError()


mm_matrix = compute_generalization_matrix(train_acts, train_labels, test_acts, test_labels, DATASET_NAMES, MMProbe)
lr_matrix = compute_generalization_matrix(train_acts, train_labels, test_acts, test_labels, DATASET_NAMES, LRProbe)

assert mm_matrix.shape == (3, 3), f"Wrong shape: {mm_matrix.shape}"
assert (mm_matrix.diag() > 0.6).all(), "In-distribution accuracy should be at least 60%"

# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["MMProbe", "LRProbe"], horizontal_spacing=0.15)

for idx, (matrix, name) in enumerate([(mm_matrix, "MM"), (lr_matrix, "LR")]):
    text_vals = [[f"{matrix[i, j]:.3f}" for j in range(len(DATASET_NAMES))] for i in range(len(DATASET_NAMES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            x=DATASET_NAMES,
            y=DATASET_NAMES,
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0.5,
            zmax=1.0,
            showscale=(idx == 1),
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Train dataset" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test dataset", row=1, col=idx + 1)

fig.update_layout(title="Cross-dataset Generalization (Test Accuracy)", height=400, width=800)
fig.show()

# Cosine similarity between probe directions
mm_directions = {name: MMProbe.from_data(train_acts[name], train_labels[name]).direction for name in DATASET_NAMES}
lr_directions = {name: LRProbe.from_data(train_acts[name], train_labels[name]).direction for name in DATASET_NAMES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions), ("LR", lr_directions)]:
    print(f"\n  {probe_name}Probe:")
    for i, n1 in enumerate(DATASET_NAMES):
        for j, n2 in enumerate(DATASET_NAMES):
            if j > i:
                d1 = directions[n1] / directions[n1].norm()
                d2 = directions[n2] / directions[n2].norm()
                print(f"    {n1} vs {n2}: {(d1 @ d2).item():.4f}")

Defines the four-example Spanish/English classification prompt and obtains the final token IDs of the encoded ` TRUE` and ` FALSE` strings. These token IDs define the next-token scores used for the baseline and interventions.


In [ ]:
# Few-shot prompt from geometry-of-truth/interventions.py. Note 3 true + 1 false examples:
# the paper calibrated this specific prompt for LLaMA-2-13B on sp_en_trans.
FEW_SHOT_PROMPT = """\
The Spanish word 'jirafa' means 'giraffe'. This statement is: TRUE
The Spanish word 'escribir' means 'to write'. This statement is: TRUE
The Spanish word 'gato' means 'cat'. This statement is: TRUE
The Spanish word 'aire' means 'silver'. This statement is: FALSE
"""

# Get token IDs for TRUE and FALSE
TRUE_ID = tokenizer.encode(" TRUE")[-1]
FALSE_ID = tokenizer.encode(" FALSE")[-1]

Evaluates Spanish/English statements absent from the prompt using the full-vocabulary next-token probability difference `P(TRUE) - P(FALSE)`, with positive scores classified as true. Accuracy, class means, and histograms establish the intervention baseline, subject to the built-in accuracy assertion. Tokenized inputs remain on CPU, so compatibility depends on model dispatch.


In [ ]:
def few_shot_evaluate(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    few_shot_prompt: str,
    true_id: int,
    false_id: int,
    batch_size: int = 32,
) -> Float[Tensor, " n"]:
    """
    Evaluate P(TRUE) - P(FALSE) for each statement using few-shot classification.

    Args:
        statements: List of statements to classify.
        model: Language model.
        tokenizer: Tokenizer.
        few_shot_prompt: The few-shot prefix prompt.
        true_id: Token ID for " TRUE".
        false_id: Token ID for " FALSE".
        batch_size: Batch size.

    Returns:
        Tensor of P(TRUE) - P(FALSE) for each statement.
    """
    result = t.zeros(len(statements))
    for i in range(0, len(statements), batch_size):
        statements_batch = statements[i:i + batch_size]
        for j in range(len(statements_batch)):
            statements_batch[j] = few_shot_prompt + statements_batch[j] + " This statement is:"
        inputs = tokenizer(statements_batch, padding=True, return_tensors="pt")
        with t.no_grad():
            outputs = model(**inputs)
        attn_mask = inputs["attention_mask"]
        batch_indices = t.arange(len(statements_batch))
        last_token_indices = attn_mask.sum(dim=1) - 1
        probs = t.softmax(outputs.logits[batch_indices, last_token_indices, :], dim=-1)
        p_true = probs[:, true_id]
        p_false = probs[:, false_id]
        diff = p_true - p_false
        result[i:i + len(statements_batch)] = diff
    return result
    raise NotImplementedError()


# Load sp_en_trans for evaluation (exclude statements used in the few-shot prompt) 
sp_df = datasets["sp_en_trans"]
sp_statements = sp_df["statement"].tolist()
sp_labels = t.tensor(sp_df["label"].values, dtype=t.float32)

# Filter out statements that appear in the few-shot prompt
sp_eval_mask = [s not in FEW_SHOT_PROMPT for s in sp_statements]
sp_eval_stmts = [s for s, m in zip(sp_statements, sp_eval_mask) if m]
sp_eval_labels = sp_labels[t.tensor(sp_eval_mask)]

p_diffs = few_shot_evaluate(sp_eval_stmts, model, tokenizer, FEW_SHOT_PROMPT, TRUE_ID, FALSE_ID)

# Compute accuracy
preds = (p_diffs > 0).float()
acc = (preds == sp_eval_labels).float().mean().item()
assert acc > 0.9, f"Few-shot accuracy too low: {acc:.3f} (expected > 0.9)"
true_mean = p_diffs[sp_eval_labels == 1].mean().item()
false_mean = p_diffs[sp_eval_labels == 0].mean().item()

print(f"Few-shot classification accuracy: {acc:.3f}")
print(f"Mean P(TRUE)-P(FALSE) for true statements:  {true_mean:.4f}")
print(f"Mean P(TRUE)-P(FALSE) for false statements: {false_mean:.4f}")

# Histogram
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=p_diffs[sp_eval_labels == 1].numpy(), name="True", marker_color="blue", opacity=0.6, nbinsx=30)
)
fig.add_trace(
    go.Histogram(x=p_diffs[sp_eval_labels == 0].numpy(), name="False", marker_color="red", opacity=0.6, nbinsx=30)
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Few-Shot Classification: P(TRUE) - P(FALSE)",
    xaxis_title="P(TRUE) - P(FALSE)",
    yaxis_title="Count",
    barmode="overlay",
    height=400,
    width=700,
)
fig.show()

Defines a hook that adds a scaled direction at fixed token indices for every sequence, preserving tensor or tuple output structure. It skips out-of-range positions and is not used by the batch-aware intervention experiment below.


In [ ]:
def make_intervention_hook(
    direction: Float[Tensor, " d_model"],
    scale: float,
    positions: list[int],
) -> callable:
    """
    Create a forward hook that adds scale * direction to hidden states at fixed positions.
    This handles both plain-tensor and tuple outputs from transformer layers.
    """

    def hook_fn(module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output

        for pos in positions:
            if 0 <= pos < hidden_states.shape[1]:
                hidden_states[:, pos, :] += scale * direction

        if isinstance(output, tuple):
            return (hidden_states,) + output[1:]
        else:
            return hidden_states

    return hook_fn

Fits an MM direction on all cities and negated-cities activations, scales it by projected class-mean separation, and tests addition/subtraction on Spanish/English statements across layers 8–14. The table and bars compare mean `P(TRUE) - P(FALSE)` with the no-intervention baseline. Queries are truncated to 512 tokens, and patch positions depend on suffix length computed with default special-token handling, which may offset the intended statement/suffix boundary.


In [ ]:
def intervention_experiment(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    direction: Float[Tensor, " d_model"],
    few_shot_prompt: str,
    true_id: int,
    false_id: int,
    intervene_layers: list[int],
    intervention: str = "none",
    batch_size: int = 32,
) -> Float[Tensor, " n"]:
    """
    Run the intervention experiment.

    Args:
        statements: Statements to evaluate.
        model: Language model.
        tokenizer: Tokenizer.
        direction: The (already scaled) truth direction vector.
        few_shot_prompt: Few-shot prefix.
        true_id: Token ID for " TRUE".
        false_id: Token ID for " FALSE".
        intervene_layers: List of layer indices to intervene at.
        intervention: "none", "add", or "subtract".
        batch_size: Batch size.

    Returns:
        P(TRUE) - P(FALSE) for each statement.
    """
    assert intervention in ["none", "add", "subtract"]

    # Determine how many tokens " This statement is:" adds
    suffix_tokens = tokenizer.encode(" This statement is:")
    len_suffix = len(suffix_tokens)

    p_diffs = []
    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        queries = [few_shot_prompt + stmt + " This statement is:" for stmt in batch]

        inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        # Register hooks for intervention
        hooks = []
        if intervention != "none":
            dir_device = direction.to(model.device)
            scale = 1.0 if intervention == "add" else -1.0

            # Each sequence in the batch can have a different length, so we iterate over batch
            # elements inside the hook, using attention_mask to find real sequence lengths.
            def make_batch_hook(dir_vec, attn_mask, scl):
                def hook_fn(module, input, output):
                    # YOUR CODE HERE - implement the batch-aware hook:
                    # 1. Extract hidden_states from output (handle tuple or plain tensor)
                    # 2. For each batch element b, find end = attn_mask[b].sum()
                    # 3. Patch at positions end - len_suffix and end - len_suffix - 1
                    # 4. Return the modified output (keeping the tuple structure if applicable)
                    if isinstance(output, tuple):
                        hidden_states = output[0]
                    else:
                        hidden_states = output
                    end = attn_mask.sum(dim=1)
                    batch_indices = t.arange(hidden_states.shape[0], device=hidden_states.device)
                    hidden_states[batch_indices, end - len_suffix - 1, :] += scl * dir_vec
                    hidden_states[batch_indices, end - len_suffix, :] += scl * dir_vec
                    if isinstance(output, tuple):
                        return (hidden_states,) + output[1:]
                    else:
                        return hidden_states
                    raise NotImplementedError()
                return hook_fn

            for layer_idx in intervene_layers:
                hook = model.model.layers[layer_idx].register_forward_hook(
                    make_batch_hook(dir_device, inputs["attention_mask"], scale)
                )
                hooks.append(hook)

        with t.no_grad():
            # Common pattern for hooks, so failed hooks don't get stuck
            try:
                outputs = model(**inputs)
            finally:
                for hook in hooks:
                    hook.remove()

            # Get logits at the last non-padding position, then get probability differences
            last_idx = inputs["attention_mask"].sum(dim=1) - 1
            batch_indices = t.arange(len(batch), device=outputs.logits.device)
            last_logits = outputs.logits[batch_indices, last_idx]
            probs = last_logits.softmax(dim=-1)
            p_diff = probs[:, true_id] - probs[:, false_id]
            p_diffs.append(p_diff.cpu().float())

    return t.cat(p_diffs)


# Train the intervention probe on cities + neg_cities combined. The paper found that
# "training on statements and their opposites improves generalization" - using both
# a statement and its negation gives the probe a cleaner truth direction.
# Load neg_cities for this paired training
neg_cities_df = pd.read_csv(GOT_DATASETS / "neg_cities.csv")
neg_cities_stmts = neg_cities_df["statement"].tolist()
neg_cities_labels = t.tensor(neg_cities_df["label"].values, dtype=t.float32)

neg_cities_acts_dict = extract_activations(neg_cities_stmts, model, tokenizer, [PROBE_LAYER])
neg_cities_acts = neg_cities_acts_dict[PROBE_LAYER]

# Train probe on cities + neg_cities combined
combined_acts = t.cat([activations["cities"], neg_cities_acts])
combined_labels = t.cat([labels_dict["cities"], neg_cities_labels])
combined_probe = MMProbe.from_data(combined_acts, combined_labels)

# Scale the direction
direction = combined_probe.direction
direction_hat = direction / direction.norm()
true_acts = combined_acts[combined_labels == 1]
false_acts = combined_acts[combined_labels == 0]
true_mean = true_acts.mean(0)
false_mean = false_acts.mean(0)
projection_diff = ((true_mean - false_mean) @ direction_hat).item()
scaled_direction = projection_diff * direction_hat

# Intervene at all layers from INTERVENE_LAYER through PROBE_LAYER. This matches
# the paper's "group (b)" hidden states that were found to be causally implicated.
intervene_layer_list = list(range(INTERVENE_LAYER, PROBE_LAYER + 1))

# Run for all 3 conditions × 2 subsets
results_intervention = {}
for intervention_type in ["none", "add", "subtract"]:
    for subset in ["true", "false"]:
        mask = sp_eval_labels == (1 if subset == "true" else 0)
        subset_stmts = [s for s, m in zip(sp_eval_stmts, mask.tolist()) if m]
        p_diffs = intervention_experiment(
            subset_stmts,
            model,
            tokenizer,
            scaled_direction,
            FEW_SHOT_PROMPT,
            TRUE_ID,
            FALSE_ID,
            intervene_layer_list,
            intervention=intervention_type,
        )
        results_intervention[(intervention_type, subset)] = p_diffs.mean().item()

# Print results
intervention_df = pd.DataFrame(
    {
        "Intervention": ["none", "add", "subtract"],
        "True Stmts (mean P_diff)": [
            f"{results_intervention[('none', 'true')]:.4f}",
            f"{results_intervention[('add', 'true')]:.4f}",
            f"{results_intervention[('subtract', 'true')]:.4f}",
        ],
        "False Stmts (mean P_diff)": [
            f"{results_intervention[('none', 'false')]:.4f}",
            f"{results_intervention[('add', 'false')]:.4f}",
            f"{results_intervention[('subtract', 'false')]:.4f}",
        ],
    }
)
print("\nIntervention results (mean P(TRUE) - P(FALSE)):")
display(intervention_df)

# Grouped bar chart
fig = go.Figure()
for subset, color in [("true", "blue"), ("false", "red")]:
    vals = [results_intervention[(interv, subset)] for interv in ["none", "add", "subtract"]]
    fig.add_trace(
        go.Bar(
            name=f"{subset.capitalize()} statements",
            x=["None", "Add", "Subtract"],
            y=vals,
            marker_color=color,
            opacity=0.7,
        )
    )
fig.update_layout(
    title="Causal Intervention: Effect on P(TRUE) - P(FALSE)",
    yaxis_title="Mean P(TRUE) - P(FALSE)",
    barmode="group",
    height=400,
    width=600,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()

Fits and scales an LR direction on the same combined data, repeats the interventions, and compares MM/LR score changes labeled NIE. Each NIE subtracts the matching no-intervention mean from add-to-false or subtract-from-true scores; positive values shift toward TRUE and negative values toward FALSE. LR coefficients are applied directly to raw hidden states despite being learned in standardized coordinates, limiting interpretation of this comparison.


In [ ]:
# Train LR probe on same data
lr_combined = LRProbe.from_data(combined_acts, combined_labels)
lr_direction = lr_combined.direction.detach()
lr_direction_hat = lr_direction / lr_direction.norm()
lr_proj_diff = ((true_mean - false_mean).float() @ lr_direction_hat).item()
lr_scaled_direction = lr_proj_diff * lr_direction_hat

# Run intervention for LR direction
lr_results = {}
for intervention_type in ["none", "add", "subtract"]:
    for subset in ["true", "false"]:
        mask = sp_eval_labels == (1 if subset == "true" else 0)
        subset_stmts = [s for s, m in zip(sp_eval_stmts, mask.tolist()) if m]
        p_diffs = intervention_experiment(
            subset_stmts,
            model,
            tokenizer,
            lr_scaled_direction,
            FEW_SHOT_PROMPT,
            TRUE_ID,
            FALSE_ID,
            intervene_layer_list,
            intervention=intervention_type,
        )
        lr_results[(intervention_type, subset)] = p_diffs.mean().item()

# Compute NIEs
mm_nie_false = results_intervention[("add", "false")] - results_intervention[("none", "false")]
mm_nie_true = results_intervention[("subtract", "true")] - results_intervention[("none", "true")]
lr_nie_false = lr_results[("add", "false")] - lr_results[("none", "false")]
lr_nie_true = lr_results[("subtract", "true")] - lr_results[("none", "true")]

nie_df = pd.DataFrame(
    {
        "Probe": ["MM", "MM", "LR", "LR"],
        "Intervention": ["Add to false", "Subtract from true", "Add to false", "Subtract from true"],
        "NIE": [f"{mm_nie_false:.4f}", f"{mm_nie_true:.4f}", f"{lr_nie_false:.4f}", f"{lr_nie_true:.4f}"],
    }
)
print("Natural Indirect Effects (NIE):")
display(nie_df)

# Side-by-side bar chart
fig = go.Figure()
fig.add_trace(
    go.Bar(
        name="MM Probe",
        x=["Add→False", "Sub→True"],
        y=[mm_nie_false, mm_nie_true],
        marker_color="blue",
        opacity=0.7,
    )
)
fig.add_trace(
    go.Bar(
        name="LR Probe",
        x=["Add→False", "Sub→True"],
        y=[lr_nie_false, lr_nie_true],
        marker_color="orange",
        opacity=0.7,
    )
)
fig.update_layout(
    title="Natural Indirect Effect: MM vs LR Probe Directions",
    yaxis_title="NIE (change in P(TRUE)-P(FALSE))",
    barmode="group",
    height=400,
    width=600,
)
fig.show()